In [1]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q datasets bitsandbytes evaluate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00


In [3]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 108.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0


In [4]:
!pip install -q langchain-core langchain-community langchain-huggingface faiss-cpu datasets bm25s[core] PyStemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [5]:
!pip uninstall -y numba jax

Found existing installation: numba 0.61.2
Uninstalling numba-0.61.2:
  Successfully uninstalled numba-0.61.2
Found existing installation: jax 0.11.1
Uninstalling jax-0.11.1:
  Successfully uninstalled jax-0.11.1


In [6]:
!rm -rf colab_llm_utils
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1561, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 1561 (delta 49), reused 55 (delta 24), pack-reused 1478 (from 2)
Receiving objects: 100% (1561/1561), 1.05 MiB | 19.46 MiB/s, done.
Resolving deltas: 100% (994/994), done.


In [7]:
import colab_llm_utils
from colab_llm_utils import configs

In [8]:
!rm -rf colab_rag
!git clone -b main https://github.com/ravy101/colab_rag.git

Cloning into 'colab_rag'...
remote: Enumerating objects: 174, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 174 (delta 4), reused 6 (delta 2), pack-reused 165 (from 1)
Receiving objects: 100% (174/174), 51.60 KiB | 6.45 MiB/s, done.
Resolving deltas: 100% (105/105), done.


In [9]:
from colab_rag import retriever

In [10]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import os
import random
import string

In [11]:
import torch, transformers, bitsandbytes, numpy
print(torch.__version__)        # 2.3.1+cu124
print(transformers.__version__) # 4.42.4
print(bitsandbytes.__version__) # 0.41.3
print(numpy.__version__)        # 1.26.4
print(torch.cuda.is_available())# True

2.11.0+cu130
5.18.0
0.50.2
2.1.3
True


In [12]:
model_config = configs.models.llama33_70b_chat
dataset_config = configs.datasets.mixeval_ff
inference_config = configs.inference.qa

inference_config['n_samples'] = 2000

special_tag = ""
short_name = model_config['model_name'].split('/')[-1]
#model_config['bnb_config'] = None
#colab_llm_utils.configs.models.mixed_quant_cfg

In [13]:
THINKING = False
FOLLOW_UP = False
VERIFY_OUTPUT = False
K_SHOT = False
RAG = False
COPY_FILES = True
progress =0

In [14]:
model_config

{'model_name': 'meta-llama/Llama-3.3-70B-Instruct',
 'hf_model_func': transformers.models.auto.modeling_auto.AutoModelForCausalLM,
 'bnb_config': BitsAndBytesConfig {
   "_load_in_4bit": true,
   "_load_in_8bit": false,
   "bnb_4bit_compute_dtype": "bfloat16",
   "bnb_4bit_quant_storage": "uint8",
   "bnb_4bit_quant_type": "nf4",
   "bnb_4bit_use_double_quant": true,
   "llm_int8_enable_fp32_cpu_offload": true,
   "llm_int8_has_fp16_weight": false,
   "llm_int8_skip_modules": null,
   "llm_int8_threshold": 6.0,
   "load_in_4bit": true,
   "load_in_8bit": false,
   "quant_method": "bitsandbytes"
 },
 'block_limit': 0}

In [15]:

special_prompt_note = ""
if THINKING:
  special_tag = "thinking"
  inference_config['max_new_tokens'] = 1000
  inference_config['temperature']=0.6
  inference_config['top_p'] = 0.95
  inference_config['top_k'] = 20
  inference_config['do_sample'] = True
  inference_config['thinking'] = True
  inference_config['terminators'].remove('\n')
  inference_config['terminators'].remove('\n\n')
  special_prompt_note = "Limit your thinking to 600 tokens or less.\n"


In [16]:

follow_up_prompt = None
if FOLLOW_UP:
  follow_up_prompt = colab_llm_utils.configs.datasets.axis_follow_up_prompt()


In [17]:

verify_prompt = None
if VERIFY_OUTPUT:
  special_tag = "verify"
  verify_prompt = colab_llm_utils.configs.prompts.VERIFICATION_PROMPT

In [18]:

k = 5
if K_SHOT:
  special_tag = "k_shot"
  examples = ""
  if dataset_config['dataset_location'] == "MixEval/MixEval":
    k_shot_dataset = concatenate_datasets([load_dataset(dataset_config['dataset_location'], "MixEval_Hard", split="multiple_choice", streaming=True),
                                           load_dataset(dataset_config['dataset_location'], "MixEval_Hard", split="free_form", streaming=True)]).shuffle(seed=123)
    example_data = k_shot_dataset.take(k)
  elif dataset_config.get("local_test") is not None:
    kshot_raw = pd.read_pickle(dataset_config.get("local_test"))
    if isinstance(kshot_raw, list):
        # Already a list of dicts!
        random.seed(123)
        example_data = random.sample(kshot_raw, k)
    elif isinstance(kshot_raw, pd.DataFrame):
        # Convert DataFrame records to list of dicts
        random.seed(123)
        example_data = random.sample(kshot_raw.to_dict(orient="records"), k)
    else:
        raise TypeError(f"Unexpected data type loaded from pickle: {type(kshot_raw)}")
  else:
    k_shot_dataset = load_dataset(dataset_config['dataset_location'], dataset_config['dataset_name'], split="validation", streaming=True)
    example_data = k_shot_dataset.take(k)
  for e in example_data:
    examples = examples + dataset_config['doc_to_text'](e) + " "
    ans = dataset_config['doc_to_ans'](e)
    if isinstance(ans, list):
      ans = ans[0]
    if dataset_config['dict_ans'] and dataset_config['dataset_location'] == "mandarjoshi/trivia_qa":
      examples = examples +  dataset_config['doc_to_ans'](e)['aliases'][0] + "\n\n******************************************\n\n"
    else:
      examples = examples +  ans + "\n\n******************************************\n\n"
else:
  trivia_examples = "Provide a short answer without explanation.\nQuestion: Which city is known for its famous 'Opera House' and 'Harbour Bridge'?\nShort Answer: Sydney.\n\n"
  examples = ""

In [19]:
print(examples)

In [20]:

if RAG:
  if COPY_FILES:
    !cp -r "/content/drive/MyDrive/rag_wiki_full/faiss_shards_fp16" /content/faiss_shards_fp16
    !cp -r "/content/drive/MyDrive/rag_wiki_full/bm25_shards" /content/bm25_shards
    !cp -r "/content/drive/MyDrive/rag_wiki_full/corpus.jsonl" /content/corpus.jsonl


  special_tag = "rag"
  print("copy complete.")
  #DB_DIR = "/content/drive/MyDrive/rag_wiki_full"
  hrag = retriever.SimpleHybridRetriever(
    "sentence-transformers/all-MiniLM-L6-v2",
    bm25s_path="/content/bm25_shards",
    faiss_path="/content/faiss_shards_fp16",
    corpus_path="/content/corpus.jsonl",
    bm25_mmap=True,
    faiss_mmap=True,
)
  mcqa_examples = ["[Source 4 | RRF Score: 0.0161 | Subject: unknown subject]\nThe word psychosis has two parts. The first part comes from psyche, which means soul in Ancient Greek. The second part is the ending '-osis', which means illness or unnatural condition. So literally, psychosis means unnatural condition of the soul.\nProvide a short answer without explanation.\nQuestion: Which city is known for its famous 'Opera House' and 'Harbour Bridge'?\nChoices: A. Brisbane.\nB. New York.\nC. Sydney.\nD. Paris.\nAnswer: C\n\n"]
  trivia_examples = ["[Source 4 | RRF Score: 0.0161 | Subject: unknown subject]\nThe word psychosis has two parts. The first part comes from psyche, which means soul in Ancient Greek. The second part is the ending '-osis', which means illness or unnatural condition. So literally, psychosis means unnatural condition of the soul.\nProvide a short answer without explanation.\nQuestion: Which city is known for its famous 'Opera House' and 'Harbour Bridge'?\nShort Answer: Sydney.\n\n"]
  if dataset_config['task_type'] == "qa":
    examples = trivia_examples
    examples = ""
    special_prompt_note = ""
  elif dataset_config['task_type'] == "multiple_choice":
    examples = mcqa_examples
    examples = ""
    special_prompt_note = ""

In [21]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

#hf_dataset = load_dataset(dataset_config['dataset_location'], dataset_config['dataset_name'], split=dataset_config['subset'], streaming=False)

In [22]:
# ==============================================================================
# DATASET LOADING LOGIC (Supports HuggingFace Datasets & Local Files)
# ==============================================================================

local_file_path = dataset_config.get("local_file")

if local_file_path is not None and os.path.exists(local_file_path):
    print(f"Loading local dataset from Drive: {local_file_path}")
    df_raw = pd.read_pickle(local_file_path)
    # Convert DataFrame records to list of dicts matching HF Dataset iteration structure
    if isinstance(df_raw, list):
        # Already a list of dicts!
        hf_dataset = df_raw
    elif isinstance(df_raw, pd.DataFrame):
        # Convert DataFrame records to list of dicts
        hf_dataset = df_raw.to_dict(orient="records")
    else:
        raise TypeError(f"Unexpected data type loaded from pickle: {type(df_raw)}")

    print(f"Total rows loaded from local file: {len(hf_dataset)}")

elif dataset_config['dataset_location'] == "lukaemon/bbh" and dataset_config['dataset_name'] == "all_special":
    print("Loading and concatenating all 25 BBH subtasks...")
    bbh_tasks = [
        'boolean_expressions', 'causal_judgement', 'date_understanding', 'disambiguation_qa',
        'dyck_languages', 'formal_fallacies', 'geometric_shapes', 'hyperbaton',
        'logical_deduction_five_objects', 'logical_deduction_seven_objects',
        'logical_deduction_three_objects', 'movie_recommendation', 'navigate',
        'object_counting', 'penguins_in_a_table', 'reasoning_about_colored_objects',
        'ruin_names', 'salient_translation_error_detection', 'snarks', 'sports_understanding',
        'temporal_sequences', 'tracking_shuffled_objects_five_objects',
        'tracking_shuffled_objects_seven_objects', 'tracking_shuffled_objects_three_objects',
        'web_of_lies'
    ]
    sub_datasets = [
        load_dataset(dataset_config['dataset_location'], task, split=dataset_config['subset'])
        for task in bbh_tasks
    ]
    hf_dataset = concatenate_datasets(sub_datasets)
    print(f"Total merged BBH rows: {len(hf_dataset)}")

else:
    # Standard loading for all other HuggingFace datasets
    hf_dataset = load_dataset(
        dataset_config['dataset_location'],
        dataset_config['dataset_name'],
        split=dataset_config['subset'],
        streaming=False
    )

test_data = hf_dataset

# Apply filtering if configured
if dataset_config.get("filter_fn") is not None and not isinstance(test_data, list):
    test_data = test_data.filter(dataset_config["filter_fn"])

# Shuffle if configured
if dataset_config.get("shuffle"):
    if isinstance(test_data, list):
        # Creates a new shuffled list to avoid modifying the original in-place
        random.seed(42)
        test_data = random.sample(test_data, len(test_data))
    else:
        test_data = test_data.shuffle(seed=42)

# Safe selection
n_samples = min(inference_config['n_samples'], len(test_data))
if isinstance(test_data, list):
    test_data = test_data[:n_samples]
else:
    test_data = test_data.select(range(n_samples))

README.md:   0%|          | 0.00/8.11k [00:00<?, ?B/s]

free-form.jsonl:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

multiple-choice.jsonl:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Generating free_form split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating multiple_choice split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [23]:
#inference_config['max_new_tokens'] = 25

In [24]:
torch.cuda.empty_cache()
#model_dtype=torch.float16
model = colab_llm_utils.model_tools.LlamaHelper(model_config, inference_config)

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/59.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [25]:
torch.cuda.empty_cache()
!nvidia-smi

Fri Oct  2 00:16:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             59W /  400W |   38478MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [26]:
test_data_list = []
for d in test_data:
  test_data_list.append(d)
  #print(doc_to_text_func(d))
  #print(doc_to_answer_func(d))
  last_prompt = dataset_config['doc_to_text'](d)

In [27]:
import os
if not os.path.exists(f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/"):
  os.makedirs(f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/")

In [ ]:
for i in range(progress, int(inference_config['n_samples']/inference_config['samples_per_file'])):

  # ---------------------------------------------------------------
  # LOAD PREVIOUSLY SAVED OUTPUTS FOR VERIFICATION MODE
  # ---------------------------------------------------------------
  if VERIFY_OUTPUT:
      input_path = (
          f"/content/drive/MyDrive/phase3/Llama/"
          f"{dataset_config['clean_name']}/"
          f"{dataset_config['dataset_name']}_{dataset_config['subset']}_"
          f"{short_name}_{int(i):04d}.pickle"
      )

      print(f"Loading saved inference file:\n{input_path}")
      df_prev = pd.read_pickle(input_path)

      print(f"Loaded {len(df_prev)} previously generated samples.")

  prompts = []
  responses = []
  blocks = []
  answers = []
  logit_outs = []
  token_outs = []
  meta = []
  t_types = []
  all_source_rows = []
  for j in range(i*inference_config['samples_per_file'], (i+1)*inference_config['samples_per_file']):

    # ===========================================================
    # VERIFICATION MODE
    # ===========================================================
    task_fam = test_data_list[j].get("task_family", "")
    if VERIFY_OUTPUT:

        row_idx = j - i * inference_config['samples_per_file']
        saved_row = df_prev.iloc[row_idx]

        original_prompt = saved_row["prompts"]
        original_response = saved_row["responses"]

        # responses appears to be stored as a list from get_sequence_logits
        if isinstance(original_response, list):
            original_response = original_response[0]

        prompt = verify_prompt.format(QUESTION=original_prompt, GENERATED_TEXT=original_response)

        output, sequence, logit_dict, deep_probs, meta_info = model.get_sequence_logits(prompt, use_beams=False, follow_up_prompt=follow_up_prompt )
    else:
      prompt = dataset_config['doc_to_text'](test_data_list[j])

      # DYNAMIC TERMINATOR SELECTION BASED ON TASK FAMILY

      if dataset_config['task_type'] == "mixed":
        if task_fam in ["mbpp", "grade-school-math", "rag"]:
            # Allow multi-line generation for code and math reasoning
            model.inference['terminators'] = [
                model.tokenizer.eos_token if hasattr(model.tokenizer, 'eos_token') else "<|endoftext|>",
                "Question:", "User:", "Human:", "<|eot_id|>"
            ]
            prompt = prompt + " (answer in less than 500 tokens)"
            # Allow up to 512 tokens for multi-step reasoning / code
            current_max_tokens = 512
            choices = []
        else:
            # Halt immediately on newline for MMLU, ARC, HellaSwag, WinoGrande
            model.inference['terminators'] = ["Question:", "Explanation:", "\n\n", "<|eot_id|>"]
            # Limit token generation for multiple choice to prevent runaways
            current_max_tokens = 128
            choices = test_data_list[j].get("source_row", {}).get("choices", [])
            print(f"choices: {choices}")

        # Override generation tokens dynamically if needed
        model.inference['max_new_tokens'] = current_max_tokens


      if RAG:
        if dataset_config['task_type'] == "multiple_choice" or dataset_config['task_type'] == "mixed" :
          prompt_full = test_data_list[j]['prompt']
          sr = test_data_list[j].get("source_row", {})
          clean_prompt = sr.get("question", prompt_full)
          print(f"Cleaned prompt: {clean_prompt}")
          #rag_result = hrag.hybrid_retrieve(test_data_list[j]['prompt'], k =5)
          rag_result = hrag.hybrid_retrieve(prompt_full, k =5)
          #rag_result = hrag.mcqa_hybrid_retrieve(test_data_list[j]['prompt'], choices, k=6, k_per_choice=2)
        else:
            rag_result = hrag.hybrid_retrieve(test_data_list[j]['prompt'], k =5)
        print(f"rag fetched: {len(rag_result)}")
        context = ""
        if len(rag_result) > 0:
          if max([r['score'] for r in rag_result]) >= .02:
            context = retriever.consolidate_context(rag_result, threshold = .02) + "\n"



        #important_instruction=special_prompt_note
        prompt =  context + prompt
      else:
        prompt = examples + special_prompt_note + prompt
      #logits, attn, mlp, block = model.logits_and_intermediates(prompt)
      if len(prompt) > 28000:
        print("Cropping *************************************")
        prompt = prompt[:28000]
      output, sequence, logit_dict, deep_probs, meta_info = model.get_sequence_logits(prompt, use_beams=False, follow_up_prompt=follow_up_prompt )
    print(prompt)
    print(f"Output: {output[0]}")
    blocks.append(deep_probs)
    token_outs.append(sequence)
    prompts.append(prompt)
    answers.append(dataset_config['doc_to_ans'](test_data_list[j]))
    model.reset_all()
    torch.cuda.empty_cache()
    responses.append(output)
    logit_outs.append(logit_dict)
    meta.append(meta_info)
    t_types.append(test_data_list[j].get("task_family", "unknown"))
    all_source_rows.append(test_data_list[j].get("source_row", {}))
    print(output)
    print(f"expected answer: {dataset_config['doc_to_ans'](test_data_list[j])}")
    print("\n**************************")
  df = pd.DataFrame({"prompts": prompts,
                     "responses": responses,
                     "ans": answers,
                     "logit_outs": logit_outs,
                     "token_outs": token_outs,
                     "blocks": blocks,
                     "meta": meta,
                     "task_family": t_types,
                     "source_row": all_source_rows})
  df.to_pickle(f"/content/drive/MyDrive/phase3/Llama/{dataset_config['clean_name']}/{dataset_config['dataset_name']}_{dataset_config['subset']}_{short_name}{special_tag}_{int(i):04d}.pickle")
  print("saving file.")
print("All done.")

[transformers] The following generation flags are not valid and may be ignored: ['top_k', 'early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Streaming output truncated to the last 5000 lines.
expected answer: ['Joseph Marie Charles dit Jacquard', 'Jacquard Joseph Marie', 'Joseph-Marie Jacquard', 'Joseph Marie Charles', '(Joseph Marie) Jacquard', 'Joseph Jacquard', 'Joseph Marie Jacquard']

**************************
Answer the following question with only the correct answer.
Question: 90 single use contacts come in 1 box and will last Pete 45 days.  Each box is $100.00 and currently 10% off.  If he buys 2 boxes of contact, how much will each pair of contacts cost?


IMPORTANT INSTRUCTIONS - Output ONLY the correct answer. DO NOT reference any context, sources or reasoning.

Answer: 
Output: $0.98
['$0.98']
expected answer: ['2']

**************************
Answer the following question with only the correct answer.
Context: Couceiro had undertaken a first incursion in 1911 with about 1,000 men, who had formed in groups of 60-70 men. They had left the town of Verín, 12 kilometers from the northern border with Spain, and had 

In [ ]:
_from google.colab import runtime
runtime.unassign()

In [ ]:
len(df.prompts.unique())